In [796]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [797]:
import sys
sys.path.append('/home/jovyan/work/base_demo')
import base_tool

In [798]:
import pandas as pd
pd.set_option('display.max_rows', 15)
pd.set_option('display.max_columns', 65)

数据介绍

bid_book_begin 集合竞价后的完整委托买入订单簿

ask_book_begin 集合竞价后的完整委托卖出订单簿

snap_list 连续竞价阶段的1s快照
    time_hms  时分秒字符串
    time_mark 毫秒级时间戳
    price_open 快照内首个成交价(无成交时为0.0)
    price_low  快照内最低成交价(无成交时为0.0)
    price_high 快照内最高成交价(无成交时为0.0)
    price_last 当日内最新成交价
     buy_trade 主动买入成交
    sell_trade 主动卖出成交
    bid_insert 委托买入挂单
    ask_insert 委托卖出挂单
    bid_cancel 委托买入撤单
    ask_cancel 委托卖出撤单

511520 511090 518880

In [ ]:
import numpy as np
from typing import List, Dict, Any, Optional

class FeatureExtractor:
    def __init__(self, snap_slice: List[Dict[str, Any]]):
        if not snap_slice:
            raise ValueError("snap_slice cannot be empty")
        self.snap_slice = snap_slice
        self.last = snap_slice[-1]
        self.bid_book = self.last.get('bid_book', [])
        self.ask_book = self.last.get('ask_book', [])
        
    @staticmethod
    def _safe_get_level(book: List[tuple], idx: int = 0) -> tuple:
        if book and len(book) > idx:
            return book[idx]
        return (np.nan, 0)
    
    @property
    def price_last(self) -> float:
        return self.last.get('price_last', np.nan)
    
    @property
    def best_bid(self) -> float:
        return self._safe_get_level(self.bid_book)[0]
    
    @property
    def best_ask(self) -> float:
        return self._safe_get_level(self.ask_book)[0]
    
    @property
    def spread(self) -> float:
        bid, ask = self.best_bid, self.best_ask
        if np.isnan(bid) or np.isnan(ask):
            return np.nan
        return ask - bid
    
    @property
    def volatility(self) -> float:
        prices = [snap['price_last'] for snap in self.snap_slice 
                  if snap.get('price_last') is not None]
        if len(prices) < 2:
            return 0.0
        mean_price = np.mean(prices)
        if mean_price == 0:
            return 0.0
        return np.std(prices) / mean_price
    
    @property
    def obi(self) -> float:
        bid_vol = self._safe_get_level(self.bid_book)[1]
        ask_vol = self._safe_get_level(self.ask_book)[1]
        total = bid_vol + ask_vol
        return (bid_vol - ask_vol) / total if total != 0 else 0.0
    
    @property
    def deep_obi(self) -> float:
        bid_vol = sum(level[1] for level in self.bid_book[:5]) if self.bid_book else 0
        ask_vol = sum(level[1] for level in self.ask_book[:5]) if self.ask_book else 0
        total = bid_vol + ask_vol
        return (bid_vol - ask_vol) / total if total != 0 else 0.0
    
    @property
    def wamp(self) -> float:
        bid_price, bid_vol = self._safe_get_level(self.bid_book)
        ask_price, ask_vol = self._safe_get_level(self.ask_book)
        numerator = bid_price * bid_vol + ask_price * ask_vol
        denominator = bid_vol + ask_vol
        if denominator == 0 or np.isnan(numerator):
            return 0.0
        return numerator / denominator
    
    @property
    def alpha01(self) -> float:
        prices = [snap.get('price_last') for snap in self.snap_slice]
        # 过滤 None
        prices = [p for p in prices if p is not None]
        if len(prices) < 2:
            return 0.0
        changes = [prices[i] - prices[i-1] for i in range(1, len(prices))]
        total_distance = sum(abs(c) for c in changes)
        if total_distance == 0:
            return 0.0
        net_distance = abs(prices[-1] - prices[0])
        return net_distance / total_distance
    
    @property
    def alpha02(self) -> float:
        prices = [snap.get('price_last') for snap in self.snap_slice]
        prices = [p for p in prices if p is not None and p > 0]
        if len(prices) < 3:
            return 0.0
        log_returns = [np.log(prices[i] / prices[i-1]) for i in range(1, len(prices))]
        accelerations = [log_returns[i] - log_returns[i-1] for i in range(1, len(log_returns))]
        return np.mean(accelerations) if accelerations else 0.0
    
    @property
    def alpha03(self) -> float:

        if not self.bid_book:
            return 0.0
        prices = np.array([level[0] for level in self.bid_book])
        volumes = np.array([level[1] for level in self.bid_book])
        idx = np.argsort(prices)[::-1]
        prices = prices[idx]
        volumes = volumes[idx]
        cum_vol = np.cumsum(volumes)
        ref_price = self.best_bid if not np.isnan(self.best_bid) else self.price_last
        offsets = np.abs(prices - ref_price)
        unique_offsets, unique_cum = [], []
        for o, c in zip(offsets, cum_vol):
            if not unique_offsets or o != unique_offsets[-1]:
                unique_offsets.append(o)
                unique_cum.append(c)
        if len(unique_offsets) < 2:
            return 0.0
        slope, _ = np.polyfit(unique_offsets, unique_cum, 1)
        return slope
    
        
    
    def extract_all(self) -> Dict[str, Any]:
        return {
            'price_last': self.price_last,
            'num_trades': self.last.get('num_trades', 0),
            'best_bid': self.best_bid,
            'best_ask': self.best_ask,
            'volatility': self.volatility,
            'spread': self.spread,
            'OBI': self.obi,
            'deep_OBI': self.deep_obi,
            'WAMP': self.wamp,
            'alpha01': self.alpha01,
            'alpha02': self.alpha02,
            'alpha03': self.alpha03,
        }

def create_feature(snap_slice: List[Dict[str, Any]]) -> Dict[str, Any]:
    return FeatureExtractor(snap_slice).extract_all()

In [800]:
import numpy as np

class TrainValidTest():
    def __init__(self, snap_list, param_dict, feature_func, y_func):
        if param_dict is not None:
            self.__dict__.update(param_dict)
        
        # 确保必要属性存在
        if not hasattr(self, 'x_window'):
            self.x_window = 1
        if not hasattr(self, 'y_window'):
            self.y_window = 1
        # 新增：为 short_window / long_window 提供默认值，避免 rolling 报错
        if not hasattr(self, 'short_window'):
            self.short_window = 5
        if not hasattr(self, 'long_window'):
            self.long_window = 20

        self.snap_list = snap_list.copy()
        self.create_feature = feature_func
        self.create_y = y_func

        # 预计算滚动均线
        prices = pd.Series([snap['price_last'] for snap in snap_list])
        self.short_ma = prices.rolling(window=self.short_window, min_periods=1).mean()
        self.long_ma = prices.rolling(window=self.long_window, min_periods=1).mean()

    def samples(self):
        feature_records = []   # 存放特征字典
        labels = []            # 存放标签（标量）
        n = len(self.snap_list)
        stride = getattr(self, 'stride', 1)

        for i in range(self.x_window, n - self.y_window, stride):
            flag, category = self.trigger(i)
            if not flag:
                continue
            # 提取特征（返回字典）
            x_dict = self.create_feature(self.snap_list[i - self.x_window:i])
            # 直接从字典获取 volatility，避免 .iloc
            volatility = x_dict.get('volatility', 0.0)
            y_val = self.create_y(
                self.snap_list[i:i + self.y_window],
                volatility, self.k_up, self.k_down, category
            )
            feature_records.append(x_dict)
            labels.append(y_val)

        if not feature_records:
            return pd.DataFrame(), pd.Series(dtype=float)

        X_all = pd.DataFrame(feature_records)
        y_all = pd.Series(labels)
        return X_all, y_all

    def trigger(self, time):
        short_curr = self.short_ma.iloc[time]
        long_curr = self.long_ma.iloc[time]
        short_prev = self.short_ma.iloc[time-1]
        long_prev = self.long_ma.iloc[time-1]

        if short_curr <= long_curr and short_prev > long_prev:
            return True, 1
        elif short_curr >= long_curr and short_prev < long_prev:
            return True, -1
        else:
            return False, 0

def samples_from_dates(dates, instrument_id, param_dict, create_feature, create_y):
    X_all_list = []
    y_all_list = []
    
    for date in dates:
        try:
            snap_list = base_tool.snap_list_load(instrument_id, date)
            if len(snap_list) < param_dict['x_window'] + param_dict['y_window']:
                print(f"{date}: 数据不足，跳过")
                continue
            tv = TrainValidTest(snap_list, param_dict, create_feature, create_y)
            X_day, y_day = tv.samples()
            X_all_list.append(X_day)
            y_all_list.append(y_day)
            print(f"{date}: 产生 {len(X_day)} 个样本")
        except Exception as e:
            print(f"{date}: 加载失败 - {e}")
    
    if X_all_list:
        X_total = pd.concat(X_all_list, axis=0, ignore_index=True)
        y_total = pd.concat(y_all_list, axis=0, ignore_index=True)
    else:
        X_total = pd.DataFrame()
        y_total = pd.Series()
    
    return X_total, y_total

def create_y(snap_slice, volatility, k_up, k_down,category):
    # 初始化突破时间索引为 None
    t_up = None
    t_down = None

    start = snap_slice[0]['price_last']
    if start is None or start == 0 or pd.isna(start):
        return pd.Series([0])

    up = start * (1 + volatility * k_up)
    down = start * (1 - volatility * k_down)

    for i in range(1, len(snap_slice)):
        price = snap_slice[i]['price_last']
        if price is None or pd.isna(price):
            continue

        if t_up is None and price >= up:
            t_up = i
        if t_down is None and price <= down:
            t_down = i

        if t_up is not None and t_down is not None:
            break

    # 根据触发情况决定标签
    if t_up is not None and t_down is not None:
        label = 1 if t_up < t_down else -1
    elif t_up is not None:
        label = 1
    elif t_down is not None:
        label = -1
    else:
        label = 0

    if category == label:
        label = 1
    else:
        label = 0

    return label





In [801]:
from collections import deque
import os
from typing import Dict, Any

class StrategyDemo():
    def __init__(self, model, param_dict={}) -> None:
        self.__dict__.update(param_dict)
        # 使用更具体的变量名
        data_file = f'/home/jovyan/work/backtest_result/{self.instrument_id}_{self.trade_ymd}_{self.name}.pkl'
        
        # 健壮的文件删除操作
        try:
            if os.path.exists(data_file):
                os.remove(data_file)
        except OSError as e:
            # 记录日志，而不是中断程序
            print(f"Warning: Could not delete file {data_file}: {e}")

        self.position_last = 0
        self.model = model
        
        # 优化1: 使用deque管理特征缓冲区
        self.feature_buffer = deque(maxlen=self.x_window)

        self.price_list = deque(maxlen=self.long_window)
        self.prev_signal = 0
        
        # 优化2: 引入增量计算所需的变量
        self.price_sum = 0.0

        return

    def on_snap(self, snap: Dict[str, Any]) -> None:
        price = snap.get('price_last') # 使用 .get() 方法更安全

        if not price: # 更Pythonic的空值检查
            return

        # 优化3: 增量更新价格总和
        if len(self.price_list) == self.x_window:
            # 如果缓冲区已满，减去即将被挤出的最旧价格
            self.price_sum -= self.price_list[0]
        
        self.price_list.append(price)
        self.price_sum += price

        if len(self.price_list) < self.x_window:
            self.position_last = 0
            self.prev_signal = 0
            return

        # 优化4: 利用增量总和计算均线，避免重复sum()
        long_ma = self.price_sum / self.long_window
        
        # 计算短期均线，由于short_window通常较小，这里影响不大
        # 但也可以用类似增量方式优化
        short_ma = sum(list(self.price_list)[-self.short_window:]) / self.short_window
        diff = short_ma - long_ma

        # 优化5: deque自动处理append和pop
        self.feature_buffer.append(snap)

        # 只有当特征缓冲区也准备好时才进行预测
        if len(self.feature_buffer) == self.x_window:
            feat_dict = create_feature(self.feature_buffer)          # 返回 dict
            features_df = pd.DataFrame([feat_dict])                  # 转换为 DataFrame
            prob = self.model.predict_proba(features_df)[:, 1][0]
        else:
            # 如果特征不足，可以设置一个默认prob或跳过
            return

        # 信号生成逻辑保持不变
        if diff > 0: # 金叉
            current_signal = 1
        elif diff < 0: # 死叉
            current_signal = -1
        else :
            current_signal = 0

        if current_signal != self.prev_signal and prob > self.threshold: 
            self.position_last = current_signal
            self.prev_signal = current_signal

        return

In [802]:
instrument_id = '518880'
trade_ymd = '20260319'

In [803]:
param_dict = {

    'instrument_id' : instrument_id,
    'trade_ymd' : trade_ymd,
    
    'short_window' : 300 ,
    'long_window' : 600 , 
    'threshold' : 0.5 ,
    'name' : 'MA_v1',

    'y_window' : 300 ,
    'stride': 1,

    'k_up': 3,
    'k_down': 3
}
param_dict['x_window'] = max(param_dict['short_window'],param_dict['long_window'])


In [804]:
import xgboost as xgb
import joblib
from sklearn.metrics import accuracy_score, classification_report

instrument_id = '511520'
train_days = 20
valid_days = 4
test_days = 5

trade_dates = ['20260202', '20260203', '20260204', '20260205', '20260206',
                '20260209', '20260210', '20260211', '20260212', '20260213',
                '20260224', '20260225', '20260226', '20260227', '20260302',
                '20260303', '20260304', '20260305', '20260306', '20260309',
                '20260310', '20260311', '20260312', '20260313', '20260316',
                '20260317', '20260318', '20260319', '20260320']
print(f"总交易日数量: {len(trade_dates)}")
print(f"交易日范围: {trade_dates[0]} ~ {trade_dates[-1]}")

总交易日数量: 29
交易日范围: 20260202 ~ 20260320


In [805]:
train_dates = trade_dates[:train_days]
valid_dates = trade_dates[train_days:train_days + valid_days]
test_dates = trade_dates[train_days + valid_days:train_days + valid_days + test_days]

print(f"训练集: {train_dates[0]} ~ {train_dates[-1]} ({len(train_dates)}天)")
print(f"验证集: {valid_dates[0]} ~ {valid_dates[-1]} ({len(valid_dates)}天)")
print(f"测试集: {test_dates[0]} ~ {test_dates[-1]} ({len(test_dates)}天)")

训练集: 20260202 ~ 20260309 (20天)
验证集: 20260310 ~ 20260313 (4天)
测试集: 20260316 ~ 20260320 (5天)


In [806]:
print("生成训练集样本...")
X_train, y_train = samples_from_dates(train_dates, instrument_id, param_dict, create_feature, create_y)
print(f"训练集样本: X={X_train.shape}, y={y_train.shape}")
if len(y_train) > 0:
    print(f"标签分布:\n{y_train.value_counts()}")

生成训练集样本...
20260202: 产生 26 个样本
20260203: 产生 23 个样本
20260204: 产生 22 个样本


20260205: 产生 14 个样本
20260206: 产生 26 个样本
20260209: 产生 27 个样本
20260210: 产生 31 个样本
20260211: 产生 20 个样本
20260212: 产生 20 个样本
20260213: 产生 20 个样本
20260224: 产生 28 个样本
20260225: 产生 14 个样本
20260226: 产生 26 个样本
20260227: 产生 23 个样本
20260302: 产生 21 个样本
20260303: 产生 27 个样本
20260304: 产生 21 个样本
20260305: 产生 21 个样本
20260306: 产生 28 个样本
20260309: 产生 27 个样本
训练集样本: X=(465, 12), y=(465,)
标签分布:
0    386
1     79
Name: count, dtype: int64


In [807]:
%%time
print("生成验证集样本...")
X_valid, y_valid = samples_from_dates(valid_dates, instrument_id, param_dict, create_feature, create_y)
print(f"验证集样本: X={X_valid.shape}, y={y_valid.shape}")
if len(y_valid) > 0:
    print(f"标签分布:\n{y_valid.value_counts()}")

生成验证集样本...
20260310: 产生 25 个样本
20260311: 产生 33 个样本
20260312: 产生 23 个样本
20260313: 产生 22 个样本
验证集样本: X=(103, 12), y=(103,)
标签分布:
0    77
1    26
Name: count, dtype: int64
CPU times: user 813 ms, sys: 5 ms, total: 818 ms
Wall time: 816 ms


In [808]:
%%time
scale_pos_weight_value = (y_train == 0).sum() / (y_train != 0).sum() if (y_train != 0).sum() > 0 else 1.0
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.01,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    scale_pos_weight=scale_pos_weight_value,
    random_state=42,
    n_jobs=-1,
    verbosity=1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=10
)

[0]	validation_0-logloss:0.69278
[10]	validation_0-logloss:0.68537
[20]	validation_0-logloss:0.67929
[30]	validation_0-logloss:0.67399
[40]	validation_0-logloss:0.66850
[50]	validation_0-logloss:0.66317
[60]	validation_0-logloss:0.65851
[70]	validation_0-logloss:0.65443
[80]	validation_0-logloss:0.65163
[90]	validation_0-logloss:0.64632
[100]	validation_0-logloss:0.64276
[110]	validation_0-logloss:0.63969
[120]	validation_0-logloss:0.63803
[130]	validation_0-logloss:0.63589
[140]	validation_0-logloss:0.63257
[150]	validation_0-logloss:0.63140
[160]	validation_0-logloss:0.62980
[170]	validation_0-logloss:0.62897
[180]	validation_0-logloss:0.62891
[190]	validation_0-logloss:0.62874
[200]	validation_0-logloss:0.62973
[210]	validation_0-logloss:0.62813
[220]	validation_0-logloss:0.62761
[230]	validation_0-logloss:0.62586
[240]	validation_0-logloss:0.62581
[250]	validation_0-logloss:0.62604
[260]	validation_0-logloss:0.62513
[270]	validation_0-logloss:0.62434
[280]	validation_0-logloss:0.62

[299]	validation_0-logloss:0.62371
CPU times: user 3.17 s, sys: 5.99 ms, total: 3.17 s
Wall time: 102 ms


,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

验证集准确率: 0.6407766990291263
分类报告:
              precision    recall  f1-score   support

      不可信(0)       0.76      0.77      0.76        77
       可信(1)       0.28      0.27      0.27        26

    accuracy                           0.64       103
   macro avg       0.52      0.52      0.52       103
weighted avg       0.64      0.64      0.64       103

In [809]:
y_pred_encoded = model.predict(X_valid)
y_pred = le.inverse_transform(y_pred_encoded)
print("验证集准确率:", accuracy_score(y_valid, y_pred))

unique_labels = sorted(set(y_valid.unique()) | set(np.unique(y_pred)))
label_names = { 0: '不可信(0)', 1: '可信(1)' }
target_names = [label_names.get(l, str(l)) for l in unique_labels]
print("分类报告:")
print(classification_report(y_valid, y_pred, labels=unique_labels, target_names=target_names))

验证集准确率: 0.6407766990291263
分类报告:
              precision    recall  f1-score   support

      不可信(0)       0.76      0.77      0.76        77
       可信(1)       0.28      0.27      0.27        26

    accuracy                           0.64       103
   macro avg       0.52      0.52      0.52       103
weighted avg       0.64      0.64      0.64       103



In [810]:
model_path = f'/home/jovyan/work/backtest_result/{instrument_id}_xgb_model.pkl'
joblib.dump(model, model_path)
print(f"模型已保存至: {model_path}")


模型已保存至: /home/jovyan/work/backtest_result/511520_xgb_model.pkl


# 测试集表现

In [811]:
%%time
print("生成测试集样本...")
X_test, y_test = samples_from_dates(test_dates, instrument_id, param_dict, create_feature, create_y)
print(f"测试集样本: X={X_test.shape}, y={y_test.shape}")

生成测试集样本...


20260316: 产生 21 个样本
20260317: 产生 24 个样本
20260318: 产生 19 个样本
20260319: 产生 21 个样本
20260320: 产生 28 个样本
测试集样本: X=(113, 12), y=(113,)
CPU times: user 1 s, sys: 6 ms, total: 1.01 s
Wall time: 1.01 s


In [812]:
import os
import sys
current_notebook_path = os.path.abspath('%pwd') 
current_dir = os.path.dirname(current_notebook_path)
parent_dir = os.path.dirname(current_dir)
utils_path = os.path.join(parent_dir, 'tools')
sys.path.append(utils_path)

In [813]:
from backtesting import backtest_multi_days
param_dict['trade_ymd'] = ''
strategy = StrategyDemo(model,param_dict)
backtest_df = backtest_multi_days(instrument_id,'20260310','20260320',strategy,param_dict)

/home/jovyan/work/backtest_result/511520_20260310_MA_v1.pkl 生成中
日期 20260310 回测完成，当日盈亏: 0.10
/home/jovyan/work/backtest_result/511520_20260311_MA_v1.pkl 生成中
日期 20260311 回测完成，当日盈亏: -3.20
/home/jovyan/work/backtest_result/511520_20260312_MA_v1.pkl 生成中
日期 20260312 回测完成，当日盈亏: -3.90
/home/jovyan/work/backtest_result/511520_20260313_MA_v1.pkl 生成中
日期 20260313 回测完成，当日盈亏: 2.10

instrument_id 511090
20260202
20260203
20260204
20260205
20260206
20260209
20260210
20260211
20260212
20260213
20260224
20260225
20260226
20260227
20260302
20260303
20260304
20260305
20260306
20260309
20260310
20260311
20260312
20260313
20260316
20260317
20260318
20260319
20260320

instrument_id 511520
20260202
20260203
20260204
20260205
20260206
20260209
20260210
20260211
20260212
20260213
20260224
20260225
20260226
20260227
20260302
20260303
20260304
20260305
20260306
20260309
20260310
20260311
20260312
20260313
20260316
20260317
20260318
20260319
20260320

instrument_id 518880
20260202
20260203
20260204
20260205
202602

In [814]:
backtest_df

,trade_ymd,order_time,order_price,total,trade,cancel,hold,profit_last,profits,maxdd,MAR,pper
0,20260310,2026-03-10 14:55:00,115.747,27,27,0,0,-0.2,0.1,4.3,0.02,0.00
1,20260311,2026-03-11 14:55:01,115.762,27,27,0,0,0.0,-3.2,3.2,-1.00,-0.12
2,20260312,2026-03-12 14:55:00,115.811,17,17,0,0,7.1,-3.9,11.0,-0.35,-0.23
3,20260313,2026-03-13 14:55:02,115.900,21,19,2,0,1.2,2.1,3.7,0.57,0.11
4,20260316,2026-03-16 14:55:00,115.794,18,17,1,0,0.5,5.2,4.4,1.18,0.31
5,20260317,2026-03-17 14:55:00,115.895,20,20,0,0,0.6,-2.7,5.2,-0.52,-0.14
6,20260318,2026-03-18 14:55:00,116.027,18,18,0,0,2.0,5.2,1.7,3.06,0.29
7,20260319,2026-03-19 14:55:00,116.170,24,24,0,0,0.7,4.5,3.4,1.32,0.19
8,20260320,2026-03-20 14:55:00,116.062,32,31,1,0,-1.4,0.3,4.0,0.08,0.01


In [815]:
from backtesting import backtest_summary
summary = backtest_summary(backtest_df)
print(summary)

{'交易天数': 9, '累计盈亏': np.float64(7.6), '最大单日盈利': 5.2, '最大单日亏损': -3.9, '盈利天数': 6, '亏损天数': 3, '平盘天数': 0, '胜率(%)': np.float64(66.67), '日均盈亏': np.float64(0.84), '盈亏比': np.float64(0.89)}
